# Ejercicio final

Escoge un dataset de regresión o clasificación (el que tu quieras) y diseña una red neuronal (de varias capas) para clasificarlo. El algoritmo de entrenamiento debe incluir algún mecanismo que monitorice el loss de validación para la selección del mejor modelo y algún método de regularización (dropout, $L_1$ o $L_2$). La red debe ser evaluada en un conjunto de test.

Si se usa regresión, recomiendo usar el MSE como función de pérdida (https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html).

Ejemplos clasificación:
- https://archive.ics.uci.edu/dataset/109/wine (se predice `class`)
- https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic (se predice `diagnosis`)

Ejemplos regresión:
- https://www.kaggle.com/datasets/camnugent/california-housing-prices (tiene una variable categórica y una variable con NAs; se predice `median_house_value`)
- https://www.kaggle.com/datasets/heptapod/uci-ml-datasets (se predice `MEDV`)
- https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html

**Nota.** La red tiene que recibir como entrada un valor numérico. Puede ser que haya datasets con features con NAs o con features categóricas. Si hay NAs, se pueden o borrar las filas/columnas o si tenéis conocimientos de machine learning, podéis imputar los valores. Si hay features categóricas, se pueden pasar a valor numérico con una asignación simple (por ejemplo, si la feature es "color" y tiene los valores "rojo", "verde" y "azul", se pueden asignar los valores 0, 1 y 2 respectivamente).


In [23]:
# ***   LIBRERÍAS   ***
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import load_wine # https://archive.ics.uci.edu/dataset/109/wine
import copy  # <--- IMPORTANTE: Necesario para copiar el estado del modelo

# ***   DEFINICIÓN DE LA RED O MODELO   ***
class Winenet(nn.Module):
    def __init__(self, hidden_dim: int = 64, dropout_p: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(13, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),        # <-- Regularización
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_p),        # <-- Regularización
            nn.Linear(hidden_dim // 2, 3)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)



# ***   DATASET Y DATALOADER   ***
class WineDataset(Dataset):
    def __init__(self):
        data = load_wine() # Cargamos el dataset de vino (178 muestras, 13 características, 3 clases)
        self.X = torch.tensor(
            data.data, dtype=torch.float32
        )  # X ahora tendrá forma (178, 13) 
        self.y = torch.tensor(
            data.target, dtype=torch.long
        )  # y tendrá las etiquetas (0, 1 o 2)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Particionar de los datos para entrenar y valuar rain_dataloader / test_dataloader / val_dataloader 
full_wine_dataset = WineDataset()
train_dataset, test_dataset, val_dataset = torch.utils.data.random_split(full_wine_dataset, [0.7, 0.2, 0.1], generator=torch.Generator().manual_seed(42))

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [24]:
# ***   MODELO, LOSS Y OPTIMIZADOR   ***

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Winenet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-2)


num_epochs = 100
loss_history = []
val_loss_history = []

# ***   INICIALIZACIÓN PARA EL MEJOR MODELO   ***
best_val_loss = float('inf') # Empezamos con un error "infinito"
best_model_state = None      # Aquí guardaremos los mejores pesos

for epoch in range(1, num_epochs + 1):  
    # ***   FASE DE ENTRENAMIENTO   ***
    model.train()
    running_loss = 0.0
    n_seen = 0
    for xB, yB in train_dataloader:
        xB, yB = xB.to(device), yB.to(device)
        optimizer.zero_grad()
        logits = model(xB)
        loss = criterion(logits, yB)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xB.size(0)
        n_seen += xB.size(0)
    
    epoch_train_loss = running_loss / n_seen
    loss_history.append(epoch_train_loss)

    # ***   FASE DE EVALUACIÓN   ***
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0
    
    with torch.no_grad():
        for xT, yT in test_dataloader:
            xT, yT = xT.to(device), yT.to(device)
            logits_test = model(xT)
            loss_t = criterion(logits_test, yT)
            test_loss += loss_t.item() * xT.size(0)
            
            preds = logits_test.argmax(dim=1)
            correct += (preds == yT).sum().item()
            total += yT.size(0)
    
    avg_test_loss = test_loss / total
    accuracy = correct / total
    
    print(f"\n--- Resultados en Test ---")
    print(f"  Loss:     {avg_test_loss:.4f}")
    print(f"  Accuracy: {accuracy * 100:.2f}%")
    

    # ***   FASE DE VALIDACIÓN   ***
    model.eval()
    running_val_loss = 0.0
    n_val_seen = 0
    with torch.no_grad():
        for xV, yV in val_dataloader:
            xV, yV = xV.to(device), yV.to(device)
            logits_val = model(xV)
            loss_v = criterion(logits_val, yV)
            running_val_loss += loss_v.item() * xV.size(0)
            n_val_seen += xV.size(0)
            
    epoch_val_loss = running_val_loss / n_val_seen
    val_loss_history.append(epoch_val_loss)

# --- LÓGICA DE GUARDADO DEL MEJOR MODELO ---
    if epoch_val_loss < best_val_loss:  # Si el error actual es menor que el mejor que conocíamos:
        best_val_loss = epoch_val_loss  # Guardamos una "foto" de los pesos actuales
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"Epoch {epoch:03d}: ¡Nuevo mejor modelo guardado! (Val Loss: {best_val_loss:.4f})")

# --- AL FINALIZAR EL ENTRENAMIENTO ---
if best_model_state is not None:   # Restauramos los mejores pesos en el modelo
    model.load_state_dict(best_model_state)
    print("\nEntrenamiento finalizado. Se ha restaurado el modelo con el menor loss de validación.")



Epoch 001: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1204)
Epoch 002: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1189)
Epoch 003: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1171)
Epoch 004: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1151)
Epoch 005: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1133)
Epoch 006: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1115)
Epoch 007: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1100)
Epoch 008: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1084)
Epoch 009: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1069)
Epoch 010: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1055)
Epoch 011: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1043)
Epoch 012: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1031)
Epoch 013: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1019)
Epoch 014: ¡Nuevo mejor modelo guardado! (Val Loss: 1.1008)
Epoch 015: ¡Nuevo mejor modelo guardado! (Val Loss: 1.0998)
Epoch 016: ¡Nuevo mejor modelo guardado! (Val Loss: 1.0988)
Epoch 017: ¡Nuevo mejor modelo guardado!